In [9]:
import os
import sys

# Third-party numerical and data handling
import numpy as np
import pandas as pd
# Visualization
import matplotlib.pyplot as plt

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader

from model_evaluation_helpers import check_device, read_preprocessed_images, ECGDataset, val_transforms, MultiHeadEfficientNet, get_probs_and_labels, compute_ranking_metrics
from sklearn.model_selection import train_test_split

In [10]:
# Check and get device
device = check_device()

try: 
    print(image_set["train_000000.png"])
except Exception as e:
    image_set = read_preprocessed_images("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/proc_images.h5")
    

label_df = pd.read_csv("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/train_final.csv", index_col=0)

image_names = list(image_set.keys())
image_ids = list([int(image_name.split(".")[0][-6:]) for image_name in image_names])
label_df = label_df.loc[label_df.index.isin(set(image_ids))]

# need to order label_df so that it has the same ordering as image_names
image_df = pd.DataFrame({
    "image_name": image_set.keys(),
})
image_df["image_id"] = image_df["image_name"].str.split(".", expand=True)[0].str[-6:].astype(int)
image_df = image_df.sort_values(by="image_id")
image_df = image_df.set_index("image_id")
# ensure ordering of label_df and image_df
label_df = label_df.loc[image_df.index]
X_train, X_test, y_train, y_test = train_test_split(image_df,
                                                    label_df,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    shuffle=True,
                                                    stratify=label_df[["CD", "MI", "AF", "STTC", "HYP"]]
                                                    )

train_image_names = X_train["image_name"].tolist()
test_image_names = X_test["image_name"].tolist()

val_dataset = ECGDataset(
    image_names = test_image_names,
    image_set = image_set,
    labels_df = y_test,
    transforms = val_transforms
)

num_workers = 0 if sys.platform == 'darwin' else 4 
print(f"Using num_workers = {num_workers}")

val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=32, 
                        shuffle=False, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

✓ MPS (Apple Silicon GPU) available

Selected device: mps
[[255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 ...
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]
 [255 255 255 ... 255 255 255]]
Using num_workers = 0


In [11]:
def evaluate_model(modelpath, modeltype, verbose=True):
    checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
    current_epoch = checkpoint["epoch"]
    if verbose:
        print(f"Loading from checkpoint, last run epoch was {current_epoch}")
        
    model = MultiHeadEfficientNet(
        num_conditions=5, 
        hidden_dim=512, 
        dropout_rate=0.3, 
        model=modeltype
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
    ranking_metrics = compute_ranking_metrics(val_labels, val_probs)
    
    return ranking_metrics

In [12]:
def evaluate_multiple_models(modeldict, modeltype="convnext"):
    """
    Takes in dictionary with format {str: str}, representing {modelname: path to model checkpoint} and collects ranking metrics
    for those models
    """
    collected_results = {}
    for modelname, modelpath in modeldict.items(): 
        results = evaluate_model(modelpath, modeltype)
        collected_results[modelname] = results
    return collected_results

In [13]:
basepath = os.path.join(os.path.expanduser("~"), "Downloads")

modeldict = {
    "Asymmetric": os.path.join(basepath, "Model_Asymm_Loss", "best_model.pth"),
    "Focal": os.path.join(basepath, "Model_Focal_Loss", "best_model.pth"), 
    "BCE_Weighted": os.path.join(basepath, "Model_BCE_Weighted", "best_model.pth"), 
    #"BCE_Unweighted": os.path.join(basepath, "Model_BCE_Unweighted", "best_model.pth")
}

agg_results = evaluate_multiple_models(modeldict)

Loading from checkpoint, last run epoch was 10


Loading from checkpoint, last run epoch was 12


Loading from checkpoint, last run epoch was 14


In [14]:
from tabulate import tabulate
models = ["Asymmetric", "Focal", "BCE_Weighted"]
to_write = []
for model in models: 
    to_write.append([model, 
                    agg_results[model]["macro_f1"],
                    agg_results[model]["macro_precision"], 
                    agg_results[model]["macro_recall"], 
                    agg_results[model]["macro_ece"], 
                    agg_results[model]["micro_ece"],
                    agg_results[model]["macro_brier"], 
                    agg_results[model]["macro_auroc"], 
                    agg_results[model]["micro_auroc"], 
                    agg_results[model]["macro_ap"], 
                    agg_results[model]["micro_ap"]])
    
print(tabulate(to_write, headers=["Model Name", "Macro F1", "Macro_Precision", "Macro_Recall", "Macro_ECE", "Micro_ECE", "Macro_Brier", "Macro_AUROC", "Micro_AUROC", "Macro_AP", "Micro_AP"]))

Model Name      Macro F1    Macro_Precision    Macro_Recall    Macro_ECE    Micro_ECE    Macro_Brier    Macro_AUROC    Micro_AUROC    Macro_AP    Micro_AP
------------  ----------  -----------------  --------------  -----------  -----------  -------------  -------------  -------------  ----------  ----------
Asymmetric      0.751339           0.745013        0.760746     0.245768    0.244243       0.136636        0.935018       0.94007     0.817444    0.822222
Focal           0.755378           0.735351        0.778831     0.096818    0.0888187      0.0783027       0.937444       0.941211    0.821188    0.813968
BCE_Weighted    0.767873           0.761198        0.776742     0.127976    0.127976       0.0914882       0.939076       0.934821    0.82112     0.79915


In [15]:
# compare per-label scores
models = ["Asymmetric", "Focal", "BCE_Weighted"]
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "f1"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_f1    HYP_f1     MI_f1     CD_f1     AF_f1
------------  ---------  --------  --------  --------  --------
Asymmetric     0.752675  0.629834  0.760956  0.766169  0.847059
Focal          0.745443  0.623776  0.75501   0.764302  0.888361
BCE_Weighted   0.758532  0.652949  0.75651   0.786667  0.884706


In [16]:
# compare per-label scores
models = ["Asymmetric", "Focal", "BCE_Weighted"]
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "precision"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_precision    HYP_precision    MI_precision    CD_precision    AF_precision
------------  ----------------  ---------------  --------------  --------------  --------------
Asymmetric            0.693735         0.644068        0.770161        0.795181        0.821918
Focal                 0.686343         0.646377        0.743949        0.730321        0.869767
BCE_Weighted          0.713075         0.662953        0.750646        0.82087         0.858447


In [17]:
# compare per-label scores
models = ["Asymmetric", "Focal", "BCE_Weighted"]
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "recall"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_recall    HYP_recall    MI_recall    CD_recall    AF_recall
------------  -------------  ------------  -----------  -----------  -----------
Asymmetric         0.822558      0.616216     0.751969       0.7392     0.873786
Focal              0.815681      0.602703     0.766404       0.8016     0.907767
BCE_Weighted       0.810179      0.643243     0.762467       0.7552     0.912621


In [18]:
# compare per-label scores
models = ["Asymmetric", "Focal", "BCE_Weighted"]
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "auroc"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name      STTC_auroc    HYP_auroc    MI_auroc    CD_auroc    AF_auroc
------------  ------------  -----------  ----------  ----------  ----------
Asymmetric        0.934665     0.903745    0.921821    0.928071    0.986786
Focal             0.933406     0.911274    0.920313    0.932163    0.990064
BCE_Weighted      0.935891     0.916787    0.921805    0.935       0.985897
